### Overview

This notebook trains ASAHI from scratch on ENCODE chr21, with early stopping, saves the model, and predicts held-out libraries with it.

The libraries are split into 6 for training, 4 for validation and 2 held out (`data/metadata/encode_demo_split.json`), with libraries from the same donor on one side. A GPU is strongly recommended. For a quick look, set `max_epochs = 3` below.

#### What happens in training
Each epoch, every chromosome patch's libraries are partitioned into slices that span the window's read-depth range. In each slice, 20% of observed cells are hidden (the imputation loss) and a nominal 20% are binomially thinned to fewer reads but stay visible (the denoising loss). Training stops when the validation denoising loss has not improved for `patience` epochs, and the best model is kept.

In [ ]:
%cd -q ..

In [ ]:
import json
import torch
from scripts.train import load_config, run_training
from scripts.inference import run_inference
from scripts.evaluate import evaluate

In [ ]:
config = load_config("example_train")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
max_epochs = None  # None: use the config's limit (100), with early stopping
print(device)
config["data"], config["train"]

### Train

In [ ]:
ckpt = run_training(config, device=device, max_epochs=max_epochs)

### Predict the 2 held-out libraries at 3x
Support rows come from the 6 training libraries at full depth.

In [ ]:
split = json.load(open(config["data"]["split_file"]))
preds = run_inference(ckpt, "chr21", "encode_pt_frac0.1_seed11", "encode_pt",
                      out="results/demo_model_frac0.1_chr21.csv",
                      anchors=split["test"], support_samples=split["train"],
                      slice_height=config["data"]["slice_height"], device=device)
per_library, summary = evaluate(preds, "chr21", "encode_pt_frac0.1_seed11", "encode_pt")
summary